1. Import Nucessary Modules

In [2]:
import re
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf 
import nltk
from bs4 import BeautifulSoup
from tensorflow.keras import models, layers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

2. Load Dataset

In [4]:
df = pd.DataFrame(pd.read_csv(r"C:\Users\hp5cd\Downloads\IMDB_dataset-1.csv"))
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


3. Text cleaning

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [ ]:
def clean(text):
    text = str(text)
    text = re.sub(r'http\S+', ' ',text)
    text = re.sub(r'@[A-Za-z0-9_]+',' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace('not good', 'bad')
    text = text.replace('not great', 'bad')
    text = text.replace('not happy', 'sad')
    text = BeautifulSoup(text, 'html.parser').get_text()
    text =  text.lower()
    return text
df['clean_review'] = df['review'].apply(clean)

C:\Users\hp5cd\AppData\Local\Temp\ipykernel_6980\1184947649.py:6: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, 'html.parser').get_text()


In [7]:
df['clean_review'].head()

0    one of the other reviewers has mentioned that ...
1    a wonderful little production. the filming tec...
2    i thought this was a wonderful way to spend ti...
3    basically there's a family where a little boy ...
4    petter mattei's "love in the time of money" is...
Name: clean_review, dtype: object

In [8]:
df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. the filming tec...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is..."


4. Split Dataset

In [9]:
X = df['clean_review']
y = df['sentiment']

In [8]:
X.head()

0    one of the other reviewers has mentioned that ...
1    a wonderful little production. the filming tec...
2    i thought this was a wonderful way to spend ti...
3    basically there's a family where a little boy ...
4    petter mattei's "love in the time of money" is...
Name: clean_review, dtype: object

In [9]:
y.head()

0    positive
1    positive
2    positive
3    negative
4    positive
Name: sentiment, dtype: object

In [10]:
y = y.map({'positive':1, 'negative':0})

In [11]:
print(len(X))
print(len(y))

50000
50000


In [12]:
# Split data into Traning and Testing data 

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.25, random_state=42)

5. Feature Extraction

In [13]:
# Conver text to word index with limited size

vocab_size = 10000
tokenize = Tokenizer(num_words = vocab_size)
tokenize.fit_on_texts(X_train)

In [14]:
# Convert word into number(index) 

X_train = tokenize.texts_to_sequences(X_train)
X_test = tokenize.texts_to_sequences(X_test)

In [15]:
# padding(add 0) with limited length 

max_len = 200
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [16]:
print(X_train.dtype)
print(y_train.dtype)

int32
int64


6. Build Model 

In [17]:
Rnn_model = models.Sequential([
    
    # Convert word index into dense vector
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),

    # RNN layer 
    layers.SimpleRNN(64),

    # Fully connected layer
    layers.Dense(64, activation='relu'),

    # Output layer (binary classification)
    layers.Dense(1, activation='sigmoid')
])

C:\Users\hp5cd\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
Rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [19]:
Rnn_model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 36s 56ms/step - accuracy: 0.6719 - loss: 0.5642 - val_accuracy: 0.8353 - val_loss: 0.3866
Epoch 2/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 36s 62ms/step - accuracy: 0.8317 - loss: 0.3870 - val_accuracy: 0.7424 - val_loss: 0.5044
Epoch 3/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.7664 - loss: 0.4766 - val_accuracy: 0.6659 - val_loss: 0.5990
Epoch 4/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 40s 68ms/step - accuracy: 0.7588 - loss: 0.4910 - val_accuracy: 0.7563 - val_loss: 0.5061
Epoch 5/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 44s 75ms/step - accuracy: 0.7648 - loss: 0.4828 - val_accuracy: 0.6570 - val_loss: 0.6257


In [20]:
Rnn_test_loss, Rnn_test_acc = Rnn_model.evaluate(X_test, y_test)
print('Rnn Test accuracy :', Rnn_test_acc)

391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.6570 - loss: 0.6257
Rnn Test accuracy : 0.657039999961853


In [21]:
Lstm_model = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    layers.LSTM(64, dropout=0.2),
    layers.Dense(1, activation='sigmoid')
])

In [22]:
Lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [23]:
Lstm_model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 80s 128ms/step - accuracy: 0.8186 - loss: 0.3894 - val_accuracy: 0.8856 - val_loss: 0.2821
Epoch 2/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 117s 188ms/step - accuracy: 0.9022 - loss: 0.2462 - val_accuracy: 0.8856 - val_loss: 0.2833
Epoch 3/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 128s 164ms/step - accuracy: 0.9252 - loss: 0.1936 - val_accuracy: 0.8818 - val_loss: 0.2872
Epoch 4/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 118s 123ms/step - accuracy: 0.9402 - loss: 0.1603 - val_accuracy: 0.8874 - val_loss: 0.2936
Epoch 5/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 74s 127ms/step - accuracy: 0.9456 - loss: 0.1429 - val_accuracy: 0.8800 - val_loss: 0.3758


In [24]:
Lstm_test_loss, Lstm_test_acc = Lstm_model.evaluate(X_test, y_test) 
print('LSTM Test Accuracy : ', Lstm_test_acc)

391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - accuracy: 0.8800 - loss: 0.3758
LSTM Test Accuracy :  0.8799999952316284


In [25]:
gru_model = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    layers.GRU(64, dropout=0.2),
    layers.Dense(1, activation='sigmoid')
])

In [26]:
gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [27]:
gru_model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 83s 135ms/step - accuracy: 0.8044 - loss: 0.4164 - val_accuracy: 0.8478 - val_loss: 0.3603
Epoch 2/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 87s 143ms/step - accuracy: 0.8991 - loss: 0.2551 - val_accuracy: 0.8908 - val_loss: 0.2668
Epoch 3/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 108s 184ms/step - accuracy: 0.9283 - loss: 0.1913 - val_accuracy: 0.9038 - val_loss: 0.2458
Epoch 4/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 122s 207ms/step - accuracy: 0.9468 - loss: 0.1496 - val_accuracy: 0.8994 - val_loss: 0.2578
Epoch 5/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 124s 212ms/step - accuracy: 0.9619 - loss: 0.1098 - val_accuracy: 0.8980 - val_loss: 0.2903


In [28]:
gru_test_loss, gru_test_acc = gru_model.evaluate(X_test, y_test)
print('GRU Test Accuracy :', gru_test_acc)

391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.8980 - loss: 0.2903
GRU Test Accuracy : 0.8980000019073486


In [29]:
bidirectional_gru_model = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    layers.Bidirectional(layers.GRU(64, dropout=0.2)),
    layers.Dense(1, 'sigmoid')
])

In [30]:
bidirectional_gru_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [31]:
bidirectional_gru_model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 200s 331ms/step - accuracy: 0.8054 - loss: 0.4115 - val_accuracy: 0.8712 - val_loss: 0.3118
Epoch 2/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 163s 265ms/step - accuracy: 0.8978 - loss: 0.2569 - val_accuracy: 0.8930 - val_loss: 0.2675
Epoch 3/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 167s 204ms/step - accuracy: 0.9262 - loss: 0.1964 - val_accuracy: 0.8995 - val_loss: 0.2517
Epoch 4/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 156s 228ms/step - accuracy: 0.9465 - loss: 0.1499 - val_accuracy: 0.8942 - val_loss: 0.2749
Epoch 5/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 172s 294ms/step - accuracy: 0.9599 - loss: 0.1154 - val_accuracy: 0.8904 - val_loss: 0.2921


In [32]:
bidirectional_gru_model_loss, bidirectional_gru_model_acc = bidirectional_gru_model.evaluate(X_test, y_test)

391/391 ━━━━━━━━━━━━━━━━━━━━ 19s 48ms/step - accuracy: 0.8904 - loss: 0.2921


In [33]:
print('RNN Test Accuracy :', Rnn_test_acc)
print('LSTM Test Accuracy :', Lstm_test_acc)
print('GRU Test Accuracy :', gru_test_acc)
print('Bidirectional GRU Test Accuracy :', bidirectional_gru_model_acc)

RNN Test Accuracy : 0.657039999961853
LSTM Test Accuracy : 0.8799999952316284
GRU Test Accuracy : 0.8980000019073486
Bidirectional GRU Test Accuracy : 0.8903999924659729


In [34]:
y_pred_prob = bidirectional_gru_model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print('Confusion Matrix :\n', confusion_matrix(y_test, y_pred))
print('Classification Report :\n', classification_report(y_test, y_pred))

391/391 ━━━━━━━━━━━━━━━━━━━━ 17s 42ms/step
Confusion Matrix :
 [[5696  554]
 [ 816 5434]]
Classification Report :
               precision    recall  f1-score   support

           0       0.87      0.91      0.89      6250
           1       0.91      0.87      0.89      6250

    accuracy                           0.89     12500
   macro avg       0.89      0.89      0.89     12500
weighted avg       0.89      0.89      0.89     12500

